In [2]:
import ollama
import json
import sys
sys.path.append('../..')
import utils

## 1) Build a Model

In [8]:
def get_completion_from_messages(messages, model="qwen2.5:7b-instruct", temperature=0):
    formatted_messages = [{"role": msg["role"], "content": msg["content"]} for msg in messages]
    try:
        response = ollama.chat(model=model, messages=formatted_messages)
        return response['message']['content'].strip()  # Ensure clean output
    except Exception as e:
        print("ERROR: Ollama API call failed:", str(e))
        return "[]"

## 2) Input Classification

In [3]:
delimiter = "####"
system_message = f"""
You will be provided with online restaurant customer queries. \
The customer query will be delimited with \
{delimiter} characters.
Classify each query into a primary category \
and a secondary category.
Provide your output in JSON format with the \
keys: primary and secondary.

Primary categories: Orders, Payments, Delivery, \
Account Management, or General Inquiry.

Orders secondary categories:
Place an order
Modify an order
Cancel an order
Order status
Missing or incorrect items

Payments secondary categories:
Payment failed
Refund request
Charge explanation
Promo code issue

Delivery secondary categories:
Delivery delay
Change delivery address
Contact delivery partner
Pickup issues

Account Management secondary categories:
Login issues
Update personal information
Loyalty program
Account security

General Inquiry secondary categories:
Menu information
Allergen or dietary questions
Restaurant hours
Feedback
Speak to a human
"""

In [5]:
food_menu = utils.load_food_menu()

In [6]:
def process_user_message(user_input, all_messages, debug=True):
    delimiter = "```"

    # =========================
    # Step 1: Input validation
    # =========================
    if not user_input or not user_input.strip():
        return "Please enter a valid food-related query.", all_messages

    if debug:
        print("Step 1: Input passed basic validation.")

    price_answer = utils.get_price_answer(user_input, food_menu)

    if price_answer:
        return price_answer, all_messages
    # =========================================
    # Step 2: Extract food categories & items
    # =========================================
    try:
        category_and_product_response = utils.find_category_and_product_only(
            user_input,
            utils.get_products_and_category()
        )
        category_and_product_list = utils.read_string_to_list(
            category_and_product_response
        )
    except Exception as e:
        category_and_product_list = []
        if debug:
            print(f"Step 2 Error: {e}")

    if debug:
        print(f"Step 2: Extracted food items → {category_and_product_list}")

    # =========================================
    # Step 3: Build menu information (Python)
    # =========================================
    if category_and_product_list:
        product_information = utils.generate_output_string(
            category_and_product_list
        )
    else:
        product_information = "No matching food items found."

    if debug:
        print("Step 3: Menu information prepared.")

    # =========================================
    # Step 4: Generate restaurant chatbot response
    # =========================================
    system_prompt = (
        "You are a friendly and professional customer service assistant for an online food restaurant. "
        "Respond clearly and politely. "
        "If menu items are mentioned, briefly describe them. "
        "If no food items are found, suggest popular dishes. "
        "Ask at most ONE relevant follow-up question related to ordering."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{delimiter}{user_input}{delimiter}"}
    ]

    if product_information:
        messages.append({
            "role": "assistant",
            "content": f"Menu details:\n{product_information}"
        })

    final_response = get_completion_from_messages(
        messages
    )

    if debug:
        print("Step 4: Restaurant response generated.")

    # =========================================
    # Step 5: Rule-based response validation
    # =========================================
    is_valid = bool(final_response and final_response.strip())

    if debug:
        print(f"Step 5: Response validation → {is_valid}")

    # =========================================
    # Step 6: Final decision
    # =========================================
    if is_valid:
        if debug:
            print("Step 6: Response approved.")
        all_messages.append({"role": "user", "content": user_input})
        all_messages.append({"role": "assistant", "content": final_response})
        all_messages[:] = all_messages[-6:]  # keep last 3 turns
        return final_response, all_messages

    else:
        if debug:
            print("Step 6: Response rejected.")
        return (
            "Sorry, I couldn't understand your food order clearly. "
            "Could you please rephrase or tell me what you'd like to order?",
            all_messages
        )

In [6]:
user_input = "Recommend a dessert"
response = process_user_message(user_input, [])
print(response)

Step 1: Input passed basic validation.
Raw Response: [{"category": "Desserts", "products": ["Chocolate Brownie", "Ice Cream Sundae"]}]
Step 2: Extracted food items → [{'category': 'Desserts', 'products': ['Chocolate Brownie', 'Ice Cream Sundae']}]
Step 3: Menu information prepared.
Step 4: Restaurant response generated.
Step 5: Response validation → True
Step 6: Response approved.
('{\n    "name": "Cheesecake",\n    "category": "Desserts",\n    "type": "Vegetarian",\n    "price": 5.99,\n    "rating": 4.8,\n    "ingredients": [\n        "Cheese",\n        "Sugar",\n        "Eggs"\n    ],\n    "description": "Classic New York style cheesecake.",\n    "availability": "Available"\n}\n\nI would recommend the **Chocolate Brownie** or the **Ice Cream Sundae**. Both are highly rated and very delicious!\n\nWhich one would you like to try, or do you have any other preferences?', [{'role': 'user', 'content': 'Recommend a dessert'}, {'role': 'assistant', 'content': '{\n    "name": "Cheesecake",\n 

In [9]:
user_input = "I want one chicken biriyani and Chocolate Brownie"
response = process_user_message(user_input, [])
print(response)

Step 1: Input passed basic validation.
Raw Response: [{"category": "Main Course", "products": ["Chicken Biryani"]}, {"category": "Desserts", "products": ["Chocolate Brownie"]}]
Step 2: Extracted food items → [{'category': 'Main Course', 'products': ['Chicken Biryani']}, {'category': 'Desserts', 'products': ['Chocolate Brownie']}]
Step 3: Menu information prepared.
Step 4: Restaurant response generated.
Step 5: Response validation → True
Step 6: Response approved.
('Your order is complete with one Chicken Biryani and a Chocolate Brownie.\n\nWould you like to add any drinks or side dishes to your order?', [{'role': 'user', 'content': 'I want one chicken biriyani and Chocolate Brownie'}, {'role': 'assistant', 'content': 'Your order is complete with one Chicken Biryani and a Chocolate Brownie.\n\nWould you like to add any drinks or side dishes to your order?'}])
